**실행 설정**

입력은 `config/paths.local.json` 또는 `SLC7A5_DATA_ROOT`로 지정합니다. 결과는 이 저장소의 `results/`에 기록합니다. 입력/실행 설명은 저장소 루트 README를 참고하세요. 이 노트북에는 기존 탐색/대체 Figure 셀이 포함되어 있으며 전체 Figure 재현 검증은 별도 단계입니다.

In [ ]:
# Repository paths; inputs are read-only, new files stay in results/.
from pathlib import Path
import os
import sys
_start = Path(os.environ.get("SLC7A5_REPO_ROOT", str(Path.cwd()))).resolve()
_repo = next((p for p in [_start, *_start.parents] if (p / "slc7a5_paths.py").is_file()), None)
if _repo is None:
    raise RuntimeError("Open this notebook inside the repository or set SLC7A5_REPO_ROOT.")
if str(_repo) not in sys.path:
    sys.path.insert(0, str(_repo))
from slc7a5_paths import artifact_path, dataset_path, input_path, output_path, output_dir, setup_notebook
setup_notebook('figure1')


In [ ]:
# Import package
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc
import seaborn as sns
import matplotlib.colors as mcolors
import scipy.sparse as sp



In [ ]:
import matplotlib as mpl

mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["pdf.use14corefonts"] = False

# Figure 1A

In [ ]:
adata=sc.read(artifact_path('MASLD_collaboration/multiomics_analysis/scRNA_v1/scRNA_monoMAC_detail.h5ad'))
adata

In [ ]:
adata.obs['state'].value_counts()

In [ ]:
adata.obs['state'] = adata.obs['state'].replace({
    'healthy': 'Healthy',
    'NAFLD': 'MASLD'})

In [ ]:
adata.obs['sample'] = adata.obs['sample'].replace({
    'Healthy1_Cd45p':'Healthy_1',
    'Healthy1_Cd45mA':'Healthy_1',
    'Healthy1_Cd45mB':'Healthy_1',
    'Healthy2_Cd45p':'Healthy_2',
    'Healthy2_Cd45m':'Healthy_2',
    'Healthy3_Cd45p':'Healthy_3',
    'Healthy3_Cd45mA':'Healthy_3',
    'Healthy3_Cd45mB':'Healthy_3',
    'Healthy4_Cd45m':'Healthy_4',
    'Healthy4_Cd45p':'Healthy_4',
    'Healthy5_Cd45p':'Healthy_5'
})

sc.pl.umap(adata, color=['sample'])

In [ ]:
adata.obs['celltype'] = adata.obs['celltype'].replace({
    'T cells': 'Lymphoid',
    'Resident NK': 'Lymphoid',
    'Mono+mono derived cells': 'Myeloid',
    'Endothelial cells': 'Endothelial',
    'Circulating NK/NKT': 'Lymphoid',
    'Macrophages': 'Myeloid',
    'Fibroblasts': 'Stellete',
    'Cholangiocytes': 'Cholangiocyte',
    'B cells': 'B cell',
    'cDC1s': 'Myeloid',
    'pDCs': 'pDC',
    'Basophils': 'Basophils',
    'Plasma cells': 'Plasma cells'
})

sc.pl.umap(adata, color=['celltype'])

In [ ]:
sc.pl.umap(
    adata,
    color='celltype',
    legend_loc='on data',
    legend_fontsize=6,
    frameon=False,
    title='Cell type prediction',
    size=10,
    show=False)
plt.savefig(output_path('02_SLC7A5_mono_Journal/05_Figure/Fig1B.pdf'), bbox_inches='tight')
plt.close()

In [ ]:
sc.pl.umap(
    adata,
    color='celltype',
    #legend_loc='on data',
    legend_fontsize=6,
    frameon=False,
    title='Cell type prediction',
    size=10,
    show=False)

plt.savefig(output_path('02_SLC7A5_mono_Journal/05_Figure/Fig1Bv2.pdf'), bbox_inches='tight')
plt.close()

In [ ]:
sc.pl.umap(
    adata,
    color='celltype',
    #legend_loc='on data',
    legend_fontsize=6,
    frameon=False,
    title='Cell type prediction',
    size=10
)
plt.savefig(output_path('02_SLC7A5_mono_Journal/05_Figure/Fig1agv2.pdf'), bbox_inches='tight')
plt.close()

In [ ]:
# Make Stacked bargraph for the comparison of cluster compositions
cluster_labels = adata.obs['celltype']
batch_labels = adata.obs['sample']
crosstab = pd.crosstab(batch_labels, cluster_labels, normalize='index')

import seaborn as sns
import matplotlib.pyplot as plt


# 그래프 그리기
crosstab.plot(kind='bar', stacked=True, figsize=(12, 5))

# 그래프 설정
plt.title('Major cell lineages in MASLD vs Normal')
plt.ylabel('Proportion')
plt.xlabel('state')
plt.legend(title='MASL vs Normal', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.savefig(output_path('02_SLC7A5_mono_Journal/05_Figure/total_state_stackedbargraph.pdf'), bbox_inches='tight', dpi=300)

In [ ]:
adata.write(output_path('02_SLC7A5_mono_Journal/03_Script/Journal_script/scRNA_monoMAC_detail.h5ad'))

# Figure1B

In [ ]:
adata=sc.read(artifact_path('02_SLC7A5_mono_Journal/03_Script/Journal_script/scRNA_monoMAC_detail.h5ad'))
adata

In [ ]:
sc.pl.umap(adata, color=['state'])

In [ ]:
sc.tl.rank_genes_groups(
    adata,
    groupby='state',
    method='wilcoxon',
    use_raw=True,
    pts=True
)

sc.pl.rank_genes_groups_dotplot(adata, n_genes=5)
deg_all = sc.get.rank_genes_groups_df(adata, group=None)
deg_all

In [ ]:
top_genes = deg_all.groupby("group")["names"].apply(lambda x: x.head(20)).explode().unique().tolist()
sc.pl.heatmap(
    adata,
    var_names=top_genes,
    groupby="state",
    use_raw=False,
    cmap="bwr",
    vmin=-3,
    vmax=3,
    swap_axes=True,
    dendrogram=False,
    show=True, 
    #return_fig=True,
    figsize=(12, 7)
)
# 직접 저장
plt.close()

In [ ]:
top_genes = deg_all.groupby("group")["names"].apply(lambda x: x.head(20)).explode().unique().tolist()
sc.pl.heatmap(
    adata,
    var_names=top_genes,
    groupby="state",
    use_raw=False,
    cmap="bwr",
    vmin=-3,
    vmax=3,
    swap_axes=True,
    dendrogram=False,
    show=False, 
    #return_fig=True,
    figsize=(9, 7)
)
# 직접 저장
plt.savefig(output_path('02_SLC7A5_mono_Journal/05_Figure/cellseq_heatmap_Fig1B.pdf'), bbox_inches='tight', dpi=300)
# /home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal/05_Figure/total_state_stackedbargraph.pdf", bbox_inches='tight', dpi=300)
plt.close()

In [ ]:
top_genes = deg_all.groupby("group")["names"].apply(lambda x: x.head(20)).explode().unique().tolist()
sc.pl.heatmap(
    adata,
    var_names=top_genes,
    groupby="state",
    use_raw=False,
    cmap="bwr",
    vmin=-3,
    vmax=3,
    swap_axes=True,
    dendrogram=False,
    show=False, 
    #return_fig=True,
    figsize=(12, 7)
)
# 직접 저장
plt.savefig(output_path('02_SLC7A5_mono_Journal/05_Figure/cellseq_heatmap_Fig1Cv1.pdf'), bbox_inches='tight', dpi=300)
# /home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal/05_Figure/total_state_stackedbargraph.pdf", bbox_inches='tight', dpi=300)
plt.close()

In [ ]:
# RPS, RPL, MT 제외 필터
top_genes = (
    deg_all
    .groupby("group")["names"]
    .apply(lambda x: x.head(100))  # 넉넉하게 뽑아서 필터 후 20개 확보
    .explode()
    .reset_index(drop=True)
)

filtered_genes = top_genes[
    ~top_genes.str.startswith(("RPS", "RPL", "MT-"))
]

top_genes_final = (
    deg_all[deg_all["names"].isin(filtered_genes)]
    .groupby("group")["names"]
    .apply(lambda x: x.head(20))
    .explode()
    .unique()
    .tolist()
)

sc.pl.heatmap(
    adata,
    var_names=top_genes_final,
    groupby="state",
    use_raw=False,
    cmap="bwr",
    vmin=-3,
    vmax=3,
    swap_axes=True,
    dendrogram=False,
    show=False,
    figsize=(9, 7)
)

plt.savefig(output_path('02_SLC7A5_mono_Journal/05_Figure/cellseq_heatmap_Fig1B.pdf'), bbox_inches='tight', dpi=300)
plt.close()

In [ ]:
# RPS, RPL, MT 제외 필터
top_genes = (
    deg_all
    .groupby("group")["names"]
    .apply(lambda x: x.head(100))  # 넉넉하게 뽑아서 필터 후 20개 확보
    .explode()
    .reset_index(drop=True)
)

filtered_genes = top_genes[
    ~top_genes.str.startswith(("RPS", "RPL", "MT-"))
]

top_genes_final = (
    deg_all[deg_all["names"].isin(filtered_genes)]
    .groupby("group")["names"]
    .apply(lambda x: x.head(20))
    .explode()
    .unique()
    .tolist()
)

sc.pl.heatmap(
    adata,
    var_names=top_genes_final,
    groupby="state",
    use_raw=False,
    cmap="bwr",
    vmin=-3,
    vmax=3,
    swap_axes=True,
    dendrogram=False,
    show=False,
    figsize=(6, 7)
)

plt.savefig(output_path('02_SLC7A5_mono_Journal/05_Figure/cellseq_heatmap_Fig1B_v2.pdf'), bbox_inches='tight', dpi=300)
plt.close()

In [ ]:
top_genes_final

In [ ]:
# RPS, RPL, MT 제외 필터
top_genes = (
    deg_all
    .groupby("group")["names"]
    .apply(lambda x: x.head(100))  # 넉넉하게 뽑아서 필터 후 20개 확보
    .explode()
    .reset_index(drop=True)
)

filtered_genes = top_genes[
    ~top_genes.str.startswith(("RPS", "RPL", "MT-"))
]

top_genes_final = (
    deg_all[deg_all["names"].isin(filtered_genes)]
    .groupby("group")["names"]
    .apply(lambda x: x.head(20))
    .explode()
    .unique()
    .tolist()
)

sc.pl.heatmap(
    adata,
    var_names=top_genes_final,
    groupby="state",
    use_raw=False,
    cmap="bwr",
    vmin=-3,
    vmax=3,
    swap_axes=True,
    dendrogram=False,
    show=False,
    figsize=(12, 7)
)

plt.savefig(output_path('02_SLC7A5_mono_Journal/05_Figure/cellseq_heatmap_Fig1Cv2.pdf'), bbox_inches='tight', dpi=300)
plt.close()

In [ ]:
deg_all.to_csv(output_path('02_SLC7A5_mono_Journal/05_Figure/scRNA_state_DEG.csv'))

# Fig 1C

In [ ]:
from scipy import stats
import matplotlib.pyplot as plt                                                                                                           
import seaborn as sns
import numpy as np                                                                                                                        
import pandas as pd

In [ ]:
from scipy import stats
import matplotlib.pyplot as plt                                                                                                           
import seaborn as sns
import numpy as np                                                                                                                        
import pandas as pd

gene = 'SLC7A5'  # 원하는 유전자명                                                                                                        

# 발현값 추출                                                                                                                             
gene_idx = list(adata.var_names).index(gene)
expr = adata.X[:, gene_idx]
if hasattr(expr, 'toarray'):                                                                                                              
    expr = expr.toarray().flatten()
else:                                                                                                                                     
    expr = np.array(expr).flatten()

df = pd.DataFrame({
    'expr':     expr,
    'state':    adata.obs['state'].values,                                                                                                
    'celltype': adata.obs['celltype'].values
})                                                                                                                                        

state_order = ['Healthy', 'MASLD']                                                                                                        
palette     = {'Healthy': '#3498DB', 'MASLD': '#E74C3C'}
celltypes   = sorted(df['celltype'].unique())                                                                                             

def sig_label(p):
    if   p < 0.001: return '***'                                                                                                          
    elif p < 0.01:  return '**'
    elif p < 0.05:  return '*'
    else:           return 'ns'                                                                                                           

n_ct = len(celltypes)                                                                                                                     
fig, axes = plt.subplots(1, n_ct, figsize=(3 * n_ct, 5), sharey=False)
if n_ct == 1:
    axes = [axes]                                                                                                                         

for ax, ct in zip(axes, celltypes):                                                                                                       
    sub = df[df['celltype'] == ct]

    sns.violinplot(
        data=sub, x='state', y='expr',
        order=state_order, palette=palette,                                                                                               
        width=0.7, linewidth=0.8,
        inner='box', cut=0, ax=ax                                                                                                         
    )           

    h_vals = sub.loc[sub['state'] == 'Healthy', 'expr'].values                                                                            
    m_vals = sub.loc[sub['state'] == 'MASLD',   'expr'].values

    if len(h_vals) > 0 and len(m_vals) > 0:                                                                                               
        _, pval = stats.mannwhitneyu(h_vals, m_vals, alternative='two-sided')
        label = sig_label(pval)                                                                                                           
    else:       
        pval, label = np.nan, 'N/A'

    y_max  = sub['expr'].max()
    y_top  = y_max * 1.08                                                                                                                 
    y_text = y_max * 1.12
    ax.plot([0, 0, 1, 1], [y_top, y_top * 1.02, y_top * 1.02, y_top],
            lw=0.8, color='black')                                                                                                        
    ax.text(0.5, y_text, f'{label}\np={pval:.2e}',
            ha='center', va='bottom', fontsize=7)                                                                                         

    ax.set_title(ct, fontsize=9, fontweight='bold')                                                                                       
    ax.set_xlabel('')
    ax.set_ylabel('log-normalized expression' if ct == celltypes[0] else '')
    ax.tick_params(axis='x', labelsize=7)                                                                                                 

fig.suptitle(f'{gene} expression per cell type (Healthy vs MASLD)',                                                                       
             fontsize=12, y=1.02)
plt.tight_layout()                                                                                                                        
#plt.savefig(f'{gene}_celltype_violinplot.pdf', bbox_inches='tight')
plt.show()  

In [ ]:
gene = 'SLC7A5'

gene_idx = list(adata.var_names).index(gene)
expr = adata.X[:, gene_idx]
if hasattr(expr, 'toarray'):                                                                                                              
    expr = expr.toarray().flatten()
else:                                                                                                                                     
    expr = np.array(expr).flatten()

df = pd.DataFrame({
    'expr':     expr,
    'state':    adata.obs['state'].values,
    'celltype': adata.obs['celltype'].values                                                                                              
})

celltypes = sorted(df['celltype'].unique())
palette   = {'Healthy': '#3498DB', 'MASLD': '#E74C3C'}

def sig_label(p):
    if   p < 0.001: return '***'                                                                                                          
    elif p < 0.01:  return '**'
    elif p < 0.05:  return '*'
    else:           return 'ns'                                                                                                           

fig, axes = plt.subplots(2, 1, figsize=(len(celltypes) * 1.2 + 1, 7),                                                                     
                         sharex=True, gridspec_kw={'hspace': 0.05})

for ax, state in zip(axes, ['Healthy', 'MASLD']):
    sub = df[df['state'] == state]                                                                                                        
    sns.violinplot(
        data=sub, x='celltype', y='expr',                                                                                                 
        order=celltypes,
        color=palette[state],                                                                                                             
        width=0.8, linewidth=0.8,
        inner='box', cut=0, ax=ax                                                                                                         
    )
    ax.set_ylabel(f'{state}\nlog-norm expr', fontsize=9)                                                                                  
    ax.set_xlabel('')                                                                                                                     
    if ax != axes[-1]:
        ax.set_xticklabels([])                                                                                                            

# celltype별 유의성 표시 (위 subplot 상단) 
ax_top = axes[0]
star_y = 4.5                                                    

for i, ct in enumerate(celltypes):                                                                                                        
    h_vals = df[(df['state'] == 'Healthy') & (df['celltype'] == ct)]['expr'].values                                                       
    m_vals = df[(df['state'] == 'MASLD')   & (df['celltype'] == ct)]['expr'].values                                                       

    if len(h_vals) > 0 and len(m_vals) > 0:                                                                                               
        _, pval = stats.mannwhitneyu(h_vals, m_vals, alternative='two-sided')
        label = sig_label(pval)                                                                                                           
    else:       
        label = 'N/A'                                                                                                                     

    ax_top.text(i, star_y, label,                                                                                                         
                ha='center', va='bottom', fontsize=8, fontweight='bold')                                                                

axes[-1].set_xticklabels(celltypes, rotation=30, ha='right', fontsize=8)                                                                  
fig.suptitle(f'{gene} expression by cell type', fontsize=12, y=1.01)

plt.savefig(output_path('02_SLC7A5_mono_Journal/05_Figure/cellseq_violinplot_Fig1C.pdf'), bbox_inches='tight', dpi=300)
plt.show()  

In [ ]:
gene = 'SLC7A5'

gene_idx = list(adata.var_names).index(gene)
expr = adata.X[:, gene_idx]
if hasattr(expr, 'toarray'):                                                                                                              
    expr = expr.toarray().flatten()
else:                                                                                                                                     
    expr = np.array(expr).flatten()

df = pd.DataFrame({
    'expr':     expr,
    'state':    adata.obs['state'].values,
    'celltype': adata.obs['celltype'].values                                                                                              
})

celltypes = sorted(df['celltype'].unique())
palette   = {'Healthy': '#3498DB', 'MASLD': '#E74C3C'}

def sig_label(p):
    if   p < 0.001: return '***'                                                                                                          
    elif p < 0.01:  return '**'
    elif p < 0.05:  return '*'
    else:           return 'ns'                                                                                                           

fig, axes = plt.subplots(2, 1, figsize=(len(celltypes) * 1.2 + 1, 7),                                                                     
                         sharex=True, gridspec_kw={'hspace': 0.05})

for ax, state in zip(axes, ['Healthy', 'MASLD']):
    sub = df[df['state'] == state]                                                                                                        
    sns.violinplot(
        data=sub, x='celltype', y='expr',                                                                                                 
        order=celltypes,
        color=palette[state],                                                                                                             
        width=0.8, linewidth=0.8,
        inner='box', cut=0, ax=ax                                                                                                         
    )
    ax.set_ylabel(f'{state}\nlog-norm expr', fontsize=9)                                                                                  
    ax.set_xlabel('')                                                                                                                     
    if ax != axes[-1]:
        ax.set_xticklabels([])                                                                                                            

# celltype별 유의성 표시 (위 subplot 상단) 
ax_top = axes[0]
star_y = 4.5                                                    

for i, ct in enumerate(celltypes):                                                                                                        
    h_vals = df[(df['state'] == 'Healthy') & (df['celltype'] == ct)]['expr'].values                                                       
    m_vals = df[(df['state'] == 'MASLD')   & (df['celltype'] == ct)]['expr'].values                                                       

    if len(h_vals) > 0 and len(m_vals) > 0:                                                                                               
        _, pval = stats.mannwhitneyu(h_vals, m_vals, alternative='two-sided')
        label = sig_label(pval)                                                                                                           
    else:       
        label = 'N/A'                                                                                                                     

    ax_top.text(i, star_y, label,                                                                                                         
                ha='center', va='bottom', fontsize=8, fontweight='bold')                                                                

axes[-1].set_xticklabels(celltypes, rotation=30, ha='right', fontsize=8)                                                                  
fig.suptitle(f'{gene} expression by cell type', fontsize=12, y=1.01)

plt.savefig(output_path('02_SLC7A5_mono_Journal/05_Figure/cellseq_violinplot_Fig1D.pdf'), bbox_inches='tight', dpi=300)
plt.show()  

In [ ]:
# ─────────────────────────────────────────────

# gene expression 추출

# ─────────────────────────────────────────────

gene = 'SLC7A5'

expr = adata[:, gene].X

if hasattr(expr, 'toarray'):

    expr = expr.toarray().flatten()

else:

    expr = np.array(expr).flatten()

df = pd.DataFrame({

    'expr': expr,

    'state': adata.obs['state'].values,

    'celltype': adata.obs['celltype'].values

})

celltypes = sorted(df['celltype'].unique())

palette = {'Healthy': '#3498DB', 'MASLD': '#E74C3C'}

# ─────────────────────────────────────────────

# significance label

# ─────────────────────────────────────────────

def sig_label(p):

    if   p < 0.001: return '***'

    elif p < 0.01:  return '**'

    elif p < 0.05:  return '*'

    else:           return 'ns'

# ─────────────────────────────────────────────

# plotting

# ─────────────────────────────────────────────

sns.set(style="white", context="paper")

fig, axes = plt.subplots(

    2, 1,

    figsize=(max(6, len(celltypes) * 0.6), 6),

    sharex=True,

    gridspec_kw={'hspace': 0.05}

)

for ax, state in zip(axes, ['Healthy', 'MASLD']):

    sub = df[df['state'] == state]

    sns.violinplot(

        data=sub,

        x='celltype',

        y='expr',

        order=celltypes,

        color=palette[state],

        width=0.8,

        linewidth=0.8,

        inner='box',

        cut=0,

        ax=ax

    )

    ax.set_ylabel(f'{state}', fontsize=9)

    ax.set_xlabel('')

    ax.spines[['top', 'right']].set_visible(False)

    if ax != axes[-1]:

        ax.set_xticklabels([])

# ─────────────────────────────────────────────

# significance annotation

# ─────────────────────────────────────────────

ax_top = axes[0]

y_max = df['expr'].max()

for i, ct in enumerate(celltypes):

    h_vals = df[(df['state'] == 'Healthy') & (df['celltype'] == ct)]['expr'].values

    m_vals = df[(df['state'] == 'MASLD')   & (df['celltype'] == ct)]['expr'].values

    if len(h_vals) > 0 and len(m_vals) > 0:

        _, pval = stats.mannwhitneyu(h_vals, m_vals, alternative='two-sided')

        label = sig_label(pval)

    else:

        label = 'NA'

    ax_top.text(

        i,

        y_max * 1.05,

        label,

        ha='center',

        va='bottom',

        fontsize=8,

        fontweight='bold'

    )

# ─────────────────────────────────────────────

# axis & title

# ─────────────────────────────────────────────

axes[-1].set_xticklabels(celltypes, rotation=30, ha='right', fontsize=8)

fig.suptitle(

    f'{gene} expression by cell type',

    fontsize=12,

    y=0.98

)

fig.tight_layout(rect=[0, 0, 1, 0.95])

# ─────────────────────────────────────────────

# save

# ─────────────────────────────────────────────

plt.savefig(

    output_path('02_SLC7A5_mono_Journal/05_Figure/cellseq_violinplot_Fig1Dv2.pdf'),

    bbox_inches='tight',

    dpi=300

)

plt.close()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from scipy import stats

matplotlib.rcParams.update({
    'font.family':      'Arial',
    'font.size':        8,
    'axes.linewidth':   0.8,
    'xtick.major.width': 0.8,
    'ytick.major.width': 0.8,
    'xtick.major.size':  3,
    'ytick.major.size':  3,
    'pdf.fonttype':     42,   # TrueType → editable in Illustrator
    'ps.fonttype':      42,
})

# ── 1. Data extraction ────────────────────────────────────────────────────────
GENE    = 'SLC7A5'
OUTBASE = output_path('02_SLC7A5_mono_Journal/05_Figure/cellseq_violinplot_Fig1Dv3')

gene_idx = list(adata.var_names).index(GENE)
expr     = adata.X[:, gene_idx]
expr     = expr.toarray().flatten() if hasattr(expr, 'toarray') else np.array(expr).flatten()

df = pd.DataFrame({
    'expr':     expr,
    'state':    adata.obs['state'].values,
    'celltype': adata.obs['celltype'].values,
})

celltypes = sorted(df['celltype'].unique())
n_ct      = len(celltypes)

# ── 2. Aesthetics ─────────────────────────────────────────────────────────────
PALETTE  = {'Healthy': '#3498DB', 'MASLD': '#E74C3C'}
STATES   = ['Healthy', 'MASLD']
FIG_W    = max(5, n_ct * 1.2 + 1)   # minimum 5 inches
FIG_H    = 7

# ── 3. Statistical labels ─────────────────────────────────────────────────────
def sig_label(p: float) -> str:
    if   p < 0.001: return '***'
    elif p < 0.01:  return '**'
    elif p < 0.05:  return '*'
    else:           return 'ns'

pvals  = {}
labels = {}
for ct in celltypes:
    h = df[(df['state'] == 'Healthy') & (df['celltype'] == ct)]['expr'].values
    m = df[(df['state'] == 'MASLD')   & (df['celltype'] == ct)]['expr'].values
    if len(h) > 1 and len(m) > 1:
        _, p = stats.mannwhitneyu(h, m, alternative='two-sided')
    else:
        p = np.nan
    pvals[ct]  = p
    labels[ct] = sig_label(p) if not np.isnan(p) else 'N/A'

# ── 4. Figure ─────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(
    2, 1,
    figsize=(FIG_W, FIG_H),
    sharex=True,
    gridspec_kw={'hspace': 0.05, 'height_ratios': [1, 1]},
)

for ax, state in zip(axes, STATES):
    sub = df[df['state'] == state]

    vp = sns.violinplot(
        data=sub,
        x='celltype', y='expr',
        order=celltypes,
        color=PALETTE[state],
        width=0.75,
        linewidth=0.25,
        inner='box',      # median + IQR box
        cut=0,            # no extrapolation beyond data range
        saturation=0.85,
        ax=ax,
    )

    for art in ax.collections:
        if hasattr(art, 'get_linewidth'):
            art.set_linewidth(0.4)   # box/whisker 선 굵기

    for line in ax.lines:
        line.set_linewidth(0.6)      # median 선 굵기
        lx = line.get_xdata()
        if len(lx) == 2:             # whisker or box → 폭 줄이기
            mid = (lx[0] + lx[1]) / 2
            half = (lx[1] - lx[0]) / 2 * 0.45   # ← 0.45 = 원래의 45% 폭
            line.set_xdata([mid - half, mid + half])

    # ── y-axis
    ax.set_ylabel(f'{state}\nlog-norm expr', fontsize=9, labelpad=4)
    ax.set_xlabel('')
    ax.yaxis.set_major_locator(ticker.MaxNLocator(integer=False, nbins=4, prune='both'))
    ax.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.1f'))

    # ── spines: keep only left + bottom
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    # ── grid (subtle)
    ax.yaxis.grid(True, linestyle='--', linewidth=0.4, color='#cccccc', zorder=0)
    ax.set_axisbelow(True)

# ── 5. Significance annotations (top subplot) ─────────────────────────────────
ax_top  = axes[0]
y_top   = ax_top.get_ylim()[1]         # dynamic: just above the violin peak
y_star  = y_top * 1.02                 # 2 % margin above top

for i, ct in enumerate(celltypes):
    lbl = labels[ct]
    color = 'black' if lbl != 'ns' else '#888888'
    ax_top.text(
        i, y_star, lbl,
        ha='center', va='bottom',
        fontsize=8, fontweight='bold',
        color=color,
    )

# lift the top y-limit to fit the asterisks
ax_top.set_ylim(top=y_star + (y_top - ax_top.get_ylim()[0]) * 0.12)

# ── 6. x-tick labels (bottom subplot only) ────────────────────────────────────
axes[-1].set_xticklabels(celltypes, rotation=30, ha='right', fontsize=8)
axes[-1].tick_params(axis='x', which='both', length=3)

# ── 7. Title & legend patch ───────────────────────────────────────────────────
fig.suptitle(f'{GENE} expression by cell type', fontsize=11, fontweight='bold', y=1.02)

legend_patches = [
    matplotlib.patches.Patch(color=PALETTE[s], label=s) for s in STATES
]
fig.legend(
    handles=legend_patches,
    loc='upper right',
    bbox_to_anchor=(1.0, 1.0),
    frameon=False,
    fontsize=8,
)

# ── 8. Save ───────────────────────────────────────────────────────────────────
for ext in ('pdf', 'png'):
    fig.savefig(
        f"{OUTBASE}.{ext}",
        bbox_inches='tight',
        dpi=300,
        transparent=(ext == 'pdf'),   # transparent bg for Illustrator
    )

plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.patches
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from scipy import stats

matplotlib.rcParams.update({
    'font.family':       'Arial',
    'font.size':         8,
    'axes.linewidth':    0.8,
    'xtick.major.width': 0.8,
    'ytick.major.width': 0.8,
    'xtick.major.size':  3,
    'ytick.major.size':  3,
    'pdf.fonttype':      42,  # TrueType → editable in Illustrator
    'ps.fonttype':       42,
})

# ── 1. Data extraction ────────────────────────────────────────────────────────
GENE    = 'SLC7A5'
OUTBASE = output_path('02_SLC7A5_mono_Journal/05_Figure/cellseq_violinplot_Fig1Dv4')

gene_idx = list(adata.var_names).index(GENE)
expr     = adata.X[:, gene_idx]
expr     = expr.toarray().flatten() if hasattr(expr, 'toarray') else np.array(expr).flatten()

df = pd.DataFrame({
    'expr':     expr,
    'state':    adata.obs['state'].values,
    'celltype': adata.obs['celltype'].values,
})

celltypes = sorted(df['celltype'].unique())
n_ct      = len(celltypes)

# ── 2. Aesthetics ─────────────────────────────────────────────────────────────
PALETTE = {'Healthy': '#3498DB', 'MASLD': '#E74C3C'}
STATES  = ['Healthy', 'MASLD']
FIG_W   = max(5, n_ct * 1.2 + 1)
FIG_H   = 7

# ── 3. Statistical labels ─────────────────────────────────────────────────────
def sig_label(p: float) -> str:
    if   p < 0.001: return '***'
    elif p < 0.01:  return '**'
    elif p < 0.05:  return '*'
    else:           return 'ns'

labels = {}
for ct in celltypes:
    h = df[(df['state'] == 'Healthy') & (df['celltype'] == ct)]['expr'].values
    m = df[(df['state'] == 'MASLD')   & (df['celltype'] == ct)]['expr'].values
    if len(h) > 1 and len(m) > 1:
        _, p = stats.mannwhitneyu(h, m, alternative='two-sided')
        labels[ct] = sig_label(p)
    else:
        labels[ct] = 'N/A'

# ── 4. Figure ─────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(
    2, 1,
    figsize=(FIG_W, FIG_H),
    sharex=True,
    gridspec_kw={'hspace': 0.05, 'height_ratios': [1, 1]},
)

for ax, state in zip(axes, STATES):
    sub = df[df['state'] == state]

    sns.violinplot(
        data=sub,
        x='celltype', y='expr',
        order=celltypes,
        color=PALETTE[state],
        width=0.75,
        linewidth=0.8,
        inner=None,       # ← clean: 내부 표시 없음
        cut=0,
        saturation=0.85,
        ax=ax,
    )

    # median 선만 얇게 수동 추가
    for i, ct in enumerate(celltypes):
        vals = sub[sub['celltype'] == ct]['expr'].values
        if len(vals) > 0:
            med = np.median(vals)
            ax.hlines(med, i - 0.08, i + 0.08,
                      colors='white', linewidth=1.2, zorder=5)

    # ── y-axis
    ax.set_ylabel(f'{state}\nlog-norm expr', fontsize=9, labelpad=4)
    ax.set_xlabel('')
    ax.yaxis.set_major_locator(ticker.MaxNLocator(integer=False, nbins=4, prune='both'))
    ax.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.1f'))

    # ── spines
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    # ── subtle grid
    ax.yaxis.grid(True, linestyle='--', linewidth=0.4, color='#cccccc', zorder=0)
    ax.set_axisbelow(True)

# ── 5. Significance annotations (top subplot) ─────────────────────────────────
ax_top = axes[0]
y_top  = ax_top.get_ylim()[1]
y_star = y_top * 1.02

for i, ct in enumerate(celltypes):
    lbl   = labels[ct]
    color = 'black' if lbl != 'ns' else '#888888'
    ax_top.text(
        i, y_star, lbl,
        ha='center', va='bottom',
        fontsize=8, fontweight='bold',
        color=color,
    )

ax_top.set_ylim(top=y_star + (y_top - ax_top.get_ylim()[0]) * 0.12)

# ── 6. x-tick labels (bottom subplot only) ────────────────────────────────────
axes[-1].set_xticklabels(celltypes, rotation=30, ha='right', fontsize=8)
axes[-1].tick_params(axis='x', which='both', length=3)

# ── 7. Title & legend ─────────────────────────────────────────────────────────
fig.suptitle(f'{GENE} expression by cell type', fontsize=11, fontweight='bold', y=1.02)

legend_patches = [
    matplotlib.patches.Patch(color=PALETTE[s], label=s) for s in STATES
]
fig.legend(
    handles=legend_patches,
    loc='upper right',
    bbox_to_anchor=(1.0, 1.0),
    frameon=False,
    fontsize=8,
)

# ── 8. Save (PDF only, transparent bg) ───────────────────────────────────────
fig.savefig(f"{OUTBASE}.pdf", bbox_inches='tight', dpi=300, transparent=True)

plt.show()

# Fig1D SLC7A5 expression violinplot

In [ ]:
adata=sc.read(artifact_path('02_SLC7A5_mono_Journal/03_Script/Journal_script/scRNA_monoMAC_detail.h5ad'))
adata

In [ ]:
sc.pl.umap(adata, color=['celltype'])

In [ ]:
adata.X

In [ ]:
gene = "SLC7A5"

celltype_col = "celltype"
condition_col = "state"

desired_celltypes = [
    "Hepatocytes",
    "Cholangiocyte",
    "Lymphoid",
    "Myeloid",
    "B cell",
    "pDC",
    "Stellete",
    "Endothelial",
    "Basophils",
    "Plasma cells"
]

desired_conditions = ["Healthy", "MASLD"]

adata_plot = adata[
    adata.obs[celltype_col].isin(desired_celltypes)
    & adata.obs[condition_col].isin(desired_conditions)
].copy()

adata_plot.obs["celltype_condition"] = (
    adata_plot.obs[celltype_col].astype(str)
    + "-"
    + adata_plot.obs[condition_col].astype(str)
)

desired_order = [
    f"{celltype}-{condition}"
    for celltype in desired_celltypes
    for condition in desired_conditions
]

existing_order = [
    group for group in desired_order
    if group in adata_plot.obs["celltype_condition"].unique()
]

adata_plot.obs["celltype_condition"] = pd.Categorical(
    adata_plot.obs["celltype_condition"],
    categories=existing_order,
    ordered=True,
)

sc.pl.violin(
    adata_plot,
    keys=gene,
    groupby="celltype_condition",
    order=existing_order,
    use_raw=True,
    stripplot=True,
    jitter=0.25,
    rotation=90,
    size=1.2,
    show=False,
)

plt.tight_layout()
plt.show()

In [ ]:
print(adata_plot)

print(adata_plot.layers.keys())

In [ ]:
print(adata_plot.raw)

print(adata_plot.X[:5, :5])

In [ ]:
gene = "SLC7A5"
celltype_col = "celltype"
condition_col = "state"

desired_celltypes = [
    "Hepatocytes",
    "Cholangiocyte",
    "Lymphoid",
    "Myeloid",
    "B cell",
    "pDC",
    "Stellete",
    "Endothelial",
    "Basophils",
    "Plasma cells"
]

desired_conditions = ["Healthy", "MASLD"]

adata_plot = adata[
    adata.obs[celltype_col].isin(desired_celltypes)
    & adata.obs[condition_col].isin(desired_conditions)
].copy()

gene_idx = adata_plot.var_names.get_loc(gene)
expr = adata_plot.X[:, gene_idx]
if sp.issparse(expr):
    expr = expr.toarray().ravel()
else:
    expr = expr.ravel()

plot_df = pd.DataFrame({
    "expression": expr,
    "celltype": adata_plot.obs[celltype_col].astype(str).values,
    "state": adata_plot.obs[condition_col].astype(str).values,
})

plot_df["celltype"] = pd.Categorical(
    plot_df["celltype"],
    categories=desired_celltypes,
    ordered=True,
)
plot_df["state"] = pd.Categorical(
    plot_df["state"],
    categories=desired_conditions,
    ordered=True,
)

plt.figure(figsize=(max(8, len(desired_celltypes) * 1.6), 5.5))

ax = sns.violinplot(
    data=plot_df,
    x="celltype",
    y="expression",
    hue="state",
    order=desired_celltypes,
    hue_order=desired_conditions,
    dodge=True,
    cut=0,
    inner="box",
    linewidth=1,
    palette={
        "Healthy": "#d62728",
        "MASLD": "#1f77b4",
    },
)

sns.stripplot(
    data=plot_df,
    x="celltype",
    y="expression",
    hue="state",
    order=desired_celltypes,
    hue_order=desired_conditions,
    dodge=True,
    jitter=0.18,
    size=1.8,
    alpha=0.35,
    palette={
        "Healthy": "#d62728",
        "MASLD": "#1f77b4",
    },
    ax=ax,
)

handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[:2], labels[:2], title="", frameon=False, loc="upper right")

ax.set_xlabel("")
ax.set_ylabel(f"{gene} expression")
ax.set_title(f"{gene} expression by cell type and condition")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.sparse as sp

from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

gene = "SLC7A5"
celltype_col = "celltype"
condition_col = "state"

desired_celltypes = [
    "Hepatocytes",
    "Cholangiocyte",
    "Lymphoid",
    "Myeloid",
    "B cell",
    "pDC",
    "Stellete",
    "Endothelial",
    "Basophils",
    "Plasma cells"
]

desired_conditions = ["Healthy", "MASLD"]

mask = (
    adata_plot.obs[celltype_col].isin(desired_celltypes)
    & adata_plot.obs[condition_col].isin(desired_conditions)
)

# Why: scaled X가 아니라 raw/log-normalized 값을 써야 해석이 맞음
expr_source = adata_plot.raw if adata_plot.raw is not None else adata_deg

if gene not in expr_source.var_names:
    raise ValueError(f"{gene} not found in expression source var_names")

gene_idx = expr_source.var_names.get_loc(gene)
expr = expr_source.X[mask, gene_idx]

if sp.issparse(expr):
    expr = expr.toarray().ravel()
else:
    expr = np.asarray(expr).ravel()

plot_df = adata_plot.obs.loc[mask, [celltype_col, condition_col]].copy()
plot_df.columns = ["celltype", "state"]
plot_df["expression"] = expr

plot_df["celltype"] = pd.Categorical(
    plot_df["celltype"],
    categories=desired_celltypes,
    ordered=True,
)
plot_df["state"] = pd.Categorical(
    plot_df["state"],
    categories=desired_conditions,
    ordered=True,
)

plot_df = plot_df.dropna(subset=["celltype", "state", "expression"]).copy()

def p_to_star(p: float) -> str:
    if pd.isna(p):
        return "NA"
    if p < 0.001:
        return "***"
    if p < 0.01:
        return "**"
    if p < 0.05:
        return "*"
    return "ns"

def direction_from_pct(healthy_pct: float, masld_pct: float) -> str:
    if pd.isna(healthy_pct) or pd.isna(masld_pct):
        return "NA"
    if healthy_pct > masld_pct:
        return "↑ H"
    if masld_pct > healthy_pct:
        return "↑ M"
    return "="

summary_rows = []
for celltype in desired_celltypes:
    sub = plot_df.loc[plot_df["celltype"] == celltype].copy()
    healthy = sub.loc[sub["state"] == "Healthy", "expression"].values
    masld = sub.loc[sub["state"] == "MASLD", "expression"].values

    healthy_nonzero = healthy[healthy > 0]
    masld_nonzero = masld[masld > 0]

    healthy_pct_expr = (healthy > 0).mean() * 100 if len(healthy) > 0 else np.nan
    masld_pct_expr = (masld > 0).mean() * 100 if len(masld) > 0 else np.nan

    healthy_mean_expr = np.mean(healthy) if len(healthy) > 0 else np.nan
    masld_mean_expr = np.mean(masld) if len(masld) > 0 else np.nan

    healthy_nonzero_mean = np.mean(healthy_nonzero) if len(healthy_nonzero) > 0 else np.nan
    masld_nonzero_mean = np.mean(masld_nonzero) if len(masld_nonzero) > 0 else np.nan

    healthy_median = np.median(healthy) if len(healthy) > 0 else np.nan
    masld_median = np.median(masld) if len(masld) > 0 else np.nan

    if len(healthy) == 0 or len(masld) == 0:
        pvalue = np.nan
    else:
        _, pvalue = mannwhitneyu(healthy, masld, alternative="two-sided")

    direction_label = direction_from_pct(healthy_pct_expr, masld_pct_expr)

    summary_rows.append(
        {
            "celltype": celltype,
            "n_healthy": len(healthy),
            "n_masld": len(masld),
            "healthy_pct_expr": healthy_pct_expr,
            "masld_pct_expr": masld_pct_expr,
            "healthy_mean_expr": healthy_mean_expr,
            "masld_mean_expr": masld_mean_expr,
            "healthy_nonzero_mean": healthy_nonzero_mean,
            "masld_nonzero_mean": masld_nonzero_mean,
            "healthy_median": healthy_median,
            "masld_median": masld_median,
            "direction_label": direction_label,
            "pvalue": pvalue,
        }
    )

stats_df = pd.DataFrame(summary_rows)

valid_mask = stats_df["pvalue"].notna()
stats_df["p_adj"] = np.nan
if valid_mask.sum() > 0:
    _, p_adj, _, _ = multipletests(
        stats_df.loc[valid_mask, "pvalue"].values,
        alpha=0.05,
        method="fdr_bh",
    )
    stats_df.loc[valid_mask, "p_adj"] = p_adj

stats_df["star"] = stats_df["p_adj"].apply(p_to_star)

display_cols = [
    "celltype",
    "n_healthy",
    "n_masld",
    "healthy_pct_expr",
    "masld_pct_expr",
    "healthy_mean_expr",
    "masld_mean_expr",
    "healthy_nonzero_mean",
    "masld_nonzero_mean",
    "healthy_median",
    "masld_median",
    "direction_label",
    "pvalue",
    "p_adj",
    "star",
]

print(stats_df[display_cols].round(4).to_string(index=False))

#plt.figure(figsize=(max(10, len(desired_celltypes) * 1.6), 6.8))
plt.figure(figsize=(14, 5))

ax = sns.violinplot(
    data=plot_df,
    x="celltype",
    y="expression",
    hue="state",
    order=desired_celltypes,
    hue_order=desired_conditions,
    dodge=True,
    cut=0,
    inner=None,
    linewidth=1,
    palette={"Healthy": "#d62728", "MASLD": "#1f77b4"},
)

plot_df_sampled = (
    plot_df.groupby(["celltype", "state"], observed=True, group_keys=False)
    .apply(lambda x: x.sample(min(len(x), 1000), random_state=42))
    .reset_index(drop=True)
)

sns.stripplot(
    data=plot_df_sampled,
    x="celltype",
    y="expression",
    hue="state",
    order=desired_celltypes,
    hue_order=desired_conditions,
    dodge=True,
    jitter=0.16,
    size=1.0,
    alpha=0.20,
    palette={"Healthy": "#d62728", "MASLD": "#1f77b4"},
    ax=ax,
)

median_df = (
    plot_df.groupby(["celltype", "state"], observed=True)["expression"]
    .median()
    .reset_index()
)

position_map = {"Healthy": -0.2, "MASLD": 0.2}
color_map = {"Healthy": "#8b1e1e", "MASLD": "#0f4c81"}

for _, row in median_df.iterrows():
    x_center = desired_celltypes.index(row["celltype"])
    x = x_center + position_map[row["state"]]
    ax.scatter(
        x,
        row["expression"],
        s=45,
        color=color_map[row["state"]],
        edgecolor="black",
        zorder=5,
    )

y_min = float(plot_df["expression"].min())
y_max = float(plot_df["expression"].max())
y_range = y_max - y_min if y_max > y_min else 1.0
ax.set_ylim(y_min, y_max + y_range * 0.42)

for i, celltype in enumerate(desired_celltypes):
    sub = plot_df.loc[plot_df["celltype"] == celltype, "expression"]
    if sub.empty:
        continue

    row = stats_df.loc[stats_df["celltype"] == celltype]
    if row.empty:
        continue

    star = row["star"].iloc[0]
    direction_label = row["direction_label"].iloc[0]
    healthy_pct = row["healthy_pct_expr"].iloc[0]
    masld_pct = row["masld_pct_expr"].iloc[0]
    healthy_mean = row["healthy_mean_expr"].iloc[0]
    masld_mean = row["masld_mean_expr"].iloc[0]

    y = sub.max() + y_range * 0.07
    x1 = i - 0.2
    x2 = i + 0.2

    ax.plot(
        [x1, x1, x2, x2],
        [y, y + y_range * 0.02, y + y_range * 0.02, y],
        lw=1,
        c="black",
    )
    ax.text(
        i,
        y + y_range * 0.025,
        f"{star} {direction_label}",
        ha="center",
        va="bottom",
        fontsize=10,
        fontweight="bold",
    )
    ax.text(
        i,
        y + y_range * 0.09,
        f"H {healthy_pct:.1f}% / M {masld_pct:.1f}%",
        ha="center",
        va="bottom",
        fontsize=8,
        color="#444444",
    )
    ax.text(
        i,
        y + y_range * 0.14,
        f"mean: {healthy_mean:.2f} / {masld_mean:.2f}",
        ha="center",
        va="bottom",
        fontsize=8,
        color="#444444",
    )

handles, labels = ax.get_legend_handles_labels()
legend_map = {}
for h, l in zip(handles, labels):
    if l not in legend_map:
        legend_map[l] = h

ax.legend(
    [legend_map["Healthy"], legend_map["MASLD"]],
    ["Healthy", "MASLD"],
    title="",
    frameon=False,
    loc="upper right",
)

ax.set_xlabel("")
ax.set_ylabel(f"{gene} expression")
ax.set_title(
    f"{gene} expression by cell type and disease group\n"
    f"Direction uses % expressing (>0), stars use BH-adjusted Mann–Whitney p-values"
)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(output_path('02_SLC7A5_mono_Journal/05_Figure/Fig1D_cellseq.pdf'), bbox_inches='tight')
plt.show()

# Fig1E. SLC7A5 high vs low DEG and GSEA

Within each target cell type, cells are split into SLC7A5 top 20% and bottom 20%, then DEG and focused GSEA are run for SLC7A5-high vs SLC7A5-low.

In [ ]:
# Fig1E setup: SLC7A5 top/bottom 20% within Hepatocytes, Stellate, and Myeloid
import re
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
import gseapy as gp
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from scipy import sparse

H5AD_PATH = artifact_path('02_SLC7A5_mono_Journal/03_Script/Journal_script/scRNA_monoMAC_detail.h5ad')
FIG_DIR = Path(output_dir('02_SLC7A5_mono_Journal/05_Figure'))
OUTDIR = Path(output_dir('02_SLC7A5_mono_Journal/02_Analysis/Fig1E_SLC7A5_top20_bottom20_DEG_GSEA'))
FIG_DIR.mkdir(parents=True, exist_ok=True)
OUTDIR.mkdir(parents=True, exist_ok=True)

GENE = "SLC7A5"
TOP_BOTTOM_PCT = 0.20
MIN_CELLS_PER_GROUP = 25
LFC_CAP = 10.0
TARGET_CELLTYPES = ["Hepatocytes", "Stellete", "Myeloid"]

adata_fig1e = sc.read_h5ad(H5AD_PATH)
print(adata_fig1e)
print("raw exists:", adata_fig1e.raw is not None)
print("celltype counts:")
display(adata_fig1e.obs["celltype"].value_counts())


In [ ]:
# Extract SLC7A5 expression from raw when available, because adata.X is scaled and can contain negative values.
def get_gene_vector(adata_obj, gene):
    gene_upper = gene.upper()
    if adata_obj.raw is not None:
        raw_var = pd.Index([str(g).upper() for g in adata_obj.raw.var_names])
        if gene_upper in raw_var:
            raw_gene = adata_obj.raw.var_names[raw_var.get_loc(gene_upper)]
            x = adata_obj.raw[:, raw_gene].X
            return np.asarray(x.toarray()).ravel() if sparse.issparse(x) else np.asarray(x).ravel()

    var = pd.Index([str(g).upper() for g in adata_obj.var_names])
    if gene_upper not in var:
        raise ValueError(f"{gene} not found in adata.var_names or adata.raw.var_names")
    x = adata_obj[:, adata_obj.var_names[var.get_loc(gene_upper)]].X
    return np.asarray(x.toarray()).ravel() if sparse.issparse(x) else np.asarray(x).ravel()

adata_fig1e.obs[f"{GENE}_expr_raw"] = get_gene_vector(adata_fig1e, GENE)

available_celltypes = set(adata_fig1e.obs["celltype"].astype(str).unique())
missing = [ct for ct in TARGET_CELLTYPES if ct not in available_celltypes]
if missing:
    print("Missing target celltype labels:", missing)
    print("Available labels:", sorted(available_celltypes))

adata_fig1e_sub = adata_fig1e[adata_fig1e.obs["celltype"].isin(TARGET_CELLTYPES)].copy()
print(adata_fig1e_sub.obs["celltype"].value_counts())


In [ ]:
# DEG: SLC7A5-high top 20% vs SLC7A5-low bottom 20% per cell type.
deg_results = {}
summary_rows = []

for ct in TARGET_CELLTYPES:
    sub = adata_fig1e_sub[adata_fig1e_sub.obs["celltype"] == ct].copy()
    n_total = sub.n_obs
    n_group = int(np.floor(n_total * TOP_BOTTOM_PCT))

    if n_group < MIN_CELLS_PER_GROUP:
        print(f"[{ct}] skip: n_total={n_total}, n_group={n_group} < {MIN_CELLS_PER_GROUP}")
        continue

    expr = sub.obs[f"{GENE}_expr_raw"].to_numpy()
    order = np.argsort(expr, kind="mergesort")

    group = np.full(n_total, "middle", dtype=object)
    group[order[:n_group]] = "SLC7A5_low"
    group[order[-n_group:]] = "SLC7A5_high"
    sub.obs["SLC7A5_top20_bottom20_group"] = pd.Categorical(
        group,
        categories=["SLC7A5_low", "middle", "SLC7A5_high"],
        ordered=True,
    )

    sub_tb = sub[sub.obs["SLC7A5_top20_bottom20_group"] != "middle"].copy()
    sub_tb.obs["SLC7A5_top20_bottom20_group"] = sub_tb.obs["SLC7A5_top20_bottom20_group"].cat.remove_unused_categories()

    sc.tl.rank_genes_groups(
        sub_tb,
        groupby="SLC7A5_top20_bottom20_group",
        groups=["SLC7A5_high"],
        reference="SLC7A5_low",
        method="wilcoxon",
        use_raw=True,
        pts=True,
    )

    deg = sc.get.rank_genes_groups_df(sub_tb, group="SLC7A5_high").rename(columns={"names": "gene"})
    deg["logfoldchanges"] = pd.to_numeric(deg["logfoldchanges"], errors="coerce").clip(-LFC_CAP, LFC_CAP)
    deg["scores"] = pd.to_numeric(deg["scores"], errors="coerce")
    deg["pvals"] = pd.to_numeric(deg["pvals"], errors="coerce")
    deg["pvals_adj"] = pd.to_numeric(deg["pvals_adj"], errors="coerce")
    deg.insert(0, "celltype", ct)

    safe_ct = re.sub(r"[^A-Za-z0-9]+", "_", ct).strip("_")
    deg_path = OUTDIR / f"DEG_{safe_ct}_SLC7A5_top20_high_vs_bottom20_low.csv"
    deg.to_csv(deg_path, index=False)
    deg_results[ct] = deg

    summary_rows.append({
        "celltype": ct,
        "total_cells": n_total,
        "n_high": int((sub_tb.obs["SLC7A5_top20_bottom20_group"] == "SLC7A5_high").sum()),
        "n_low": int((sub_tb.obs["SLC7A5_top20_bottom20_group"] == "SLC7A5_low").sum()),
        "low_expr_max": float(expr[order[:n_group]].max()),
        "high_expr_min": float(expr[order[-n_group:]].min()),
        "deg_csv": str(deg_path),
        "logfc_na": int(deg["logfoldchanges"].isna().sum()),
        "sig_deg_fdr_0_05": int((deg["pvals_adj"] < 0.05).sum()),
    })

    print(f"[{ct}] DEG saved: {deg_path.name} | n/group={n_group} | logFC NaN={deg['logfoldchanges'].isna().sum()}")

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(OUTDIR / "Fig1E_SLC7A5_top20_bottom20_DEG_summary.csv", index=False)
display(summary_df)


In [ ]:
# Focused GSEA for Glycolysis, TCA cycle, and NF-kB-related pathways.
GENE_SET_COLLECTIONS = {
    "hallmark": "MSigDB_Hallmark_2020",
    "kegg": "KEGG_2021_Human",
}
FOCUS_PATTERNS = {
    "Glycolysis": r"GLYCOLYSIS|GLUCONEOGENESIS|GLUCOSE",
    "TCA cycle": r"TCA|CITRATE_CYCLE|TRICARBOXYLIC|OXIDATIVE_PHOSPHORYLATION|RESPIRATORY_CHAIN",
    "NF-kB": r"NFKB|NF_KB|TNFA_SIGNALING_VIA_NFKB|TOLL_LIKE|INFLAMMATORY_RESPONSE",
}

def make_prerank(deg):
    rank_df = deg[["gene", "logfoldchanges", "pvals", "scores"]].copy()
    rank_df["gene"] = rank_df["gene"].astype(str).str.upper()
    rank_df["pvals"] = pd.to_numeric(rank_df["pvals"], errors="coerce").clip(lower=1e-300)
    rank_df["logfoldchanges"] = pd.to_numeric(rank_df["logfoldchanges"], errors="coerce")
    rank_df["scores"] = pd.to_numeric(rank_df["scores"], errors="coerce")
    rank_df["rank_metric"] = np.sign(rank_df["logfoldchanges"]) * -np.log10(rank_df["pvals"])
    rank_df.loc[rank_df["rank_metric"].isna(), "rank_metric"] = rank_df.loc[rank_df["rank_metric"].isna(), "scores"]
    rank_df = rank_df.replace([np.inf, -np.inf], np.nan).dropna(subset=["gene", "rank_metric"])
    rank_df = rank_df.sort_values("rank_metric", ascending=False).drop_duplicates("gene", keep="first")
    return rank_df[["gene", "rank_metric"]]

gsea_results = {}
focused_rows = []

for ct, deg in deg_results.items():
    safe_ct = re.sub(r"[^A-Za-z0-9]+", "_", ct).strip("_")
    prerank = make_prerank(deg)
    prerank.to_csv(OUTDIR / f"Prerank_{safe_ct}_SLC7A5_top20_high_vs_low.rnk", sep="	", index=False, header=False)

    for label, gene_sets in GENE_SET_COLLECTIONS.items():
        pre_res = gp.prerank(
            rnk=prerank,
            gene_sets=gene_sets,
            min_size=5,
            max_size=500,
            permutation_num=1000,
            seed=7,
            outdir=None,
            verbose=False,
        )

        res = pre_res.res2d.copy()
        res.insert(0, "celltype", ct)
        res.insert(1, "gene_set", label)
        res_path = OUTDIR / f"GSEA_{safe_ct}_{label}_SLC7A5_top20_high_vs_low.csv"
        res.to_csv(res_path, index=False)
        gsea_results[(ct, label)] = res

        term_norm = res["Term"].astype(str).str.upper().str.replace(" ", "_", regex=False).str.replace("-", "_", regex=False)
        for pathway_group, pattern in FOCUS_PATTERNS.items():
            focus = res.loc[term_norm.str.contains(pattern, regex=True, na=False)].copy()
            if focus.empty:
                continue
            focus.insert(2, "pathway_group", pathway_group)
            focused_rows.append(focus)

        n_focus = sum(term_norm.str.contains(pattern, regex=True, na=False).sum() for pattern in FOCUS_PATTERNS.values())
        print(f"[{ct} | {label}] GSEA saved: {res_path.name} | focused matches={n_focus}")

focused_gsea_df = pd.concat(focused_rows, ignore_index=True) if focused_rows else pd.DataFrame()
focused_gsea_df.to_csv(OUTDIR / "GSEA_focused_glycolysis_TCA_NFKB_all_matches.csv", index=False)
display(focused_gsea_df.head(30))


In [ ]:
# Fig1E focused NES heatmap: one strongest matched term per pathway group and cell type.
if focused_gsea_df.empty:
    raise ValueError("No focused GSEA terms matched Glycolysis, TCA cycle, or NF-kB patterns.")

plot_rows = []
for (ct, pathway_group), sub in focused_gsea_df.groupby(["celltype", "pathway_group"], observed=True):
    sub = sub.copy()
    sub["abs_nes"] = pd.to_numeric(sub["NES"], errors="coerce").abs()
    best = sub.sort_values(["abs_nes", "FDR q-val"], ascending=[False, True]).iloc[0]
    plot_rows.append(best)

plot_df = pd.DataFrame(plot_rows)
plot_df.to_csv(OUTDIR / "GSEA_focused_glycolysis_TCA_NFKB_best_terms_for_heatmap.csv", index=False)

pathway_order = ["Glycolysis", "TCA cycle", "NF-kB"]
celltype_order = [ct for ct in TARGET_CELLTYPES if ct in plot_df["celltype"].unique()]
nes_mat = pd.DataFrame(np.nan, index=pathway_order, columns=celltype_order)
label_mat = pd.DataFrame("", index=pathway_order, columns=celltype_order)

for _, row in plot_df.iterrows():
    ct = row["celltype"]
    pg = row["pathway_group"]
    if pg not in nes_mat.index or ct not in nes_mat.columns:
        continue
    nes_mat.loc[pg, ct] = float(row["NES"])
    p = float(row["NOM p-val"])
    fdr = float(row["FDR q-val"])
    star = "**" if p < 0.01 else "*" if p < 0.05 else ""
    label_mat.loc[pg, ct] = f"{star}\n{row['gene_set']}: {row['Term']}\nFDR={fdr:.3g}"

vmax = np.nanmax(np.abs(nes_mat.to_numpy(dtype=float)))
vmax = 1 if not np.isfinite(vmax) or vmax == 0 else max(1, vmax)

fig, ax = plt.subplots(figsize=(max(5.5, 1.8 * len(celltype_order)), 3.8))
im = ax.imshow(
    nes_mat.to_numpy(dtype=float),
    cmap="RdBu_r",
    norm=mcolors.TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax),
    aspect="auto",
)

ax.set_xticks(range(len(celltype_order)))
ax.set_xticklabels(celltype_order, rotation=35, ha="right")
ax.set_yticks(range(len(pathway_order)))
ax.set_yticklabels(pathway_order)
ax.set_title("Fig1E. Focused GSEA: SLC7A5 top 20% high vs bottom 20% low")

for i, pathway in enumerate(pathway_order):
    for j, ct in enumerate(celltype_order):
        if pd.notna(nes_mat.loc[pathway, ct]):
            ax.text(j, i, label_mat.loc[pathway, ct], ha="center", va="center", fontsize=7)

cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label("NES (SLC7A5 high vs low)")
plt.tight_layout()

fig_pdf = FIG_DIR / "Fig1E_SLC7A5_top20_bottom20_focused_GSEA_heatmap.pdf"
fig_png = FIG_DIR / "Fig1E_SLC7A5_top20_bottom20_focused_GSEA_heatmap.png"
plt.savefig(fig_pdf, bbox_inches="tight", dpi=300)
plt.savefig(fig_png, bbox_inches="tight", dpi=200)
plt.show()

print(f"Saved -> {fig_pdf}")
print(f"Saved -> {fig_png}")
print(f"DEG/GSEA tables -> {OUTDIR}")


## Supplementary QC. SLC7A5 stratification feasibility across cell types

This QC panel evaluates whether SLC7A5 high/low stratification is robust enough for downstream DEG/GSEA in hepatocytes, stellate cells, and myeloid cells.

In [ ]:
# Supplementary QC setup: evaluate SLC7A5 high/low feasibility in Hepatocytes, Stellate, and Myeloid.
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import sparse

H5AD_PATH_QC = artifact_path('02_SLC7A5_mono_Journal/03_Script/Journal_script/scRNA_monoMAC_detail.h5ad')
FIG_DIR_QC = Path(output_dir('02_SLC7A5_mono_Journal/05_Figure'))
OUTDIR_QC = Path(output_dir('02_SLC7A5_mono_Journal/02_Analysis/SLC7A5_stratification_feasibility_QC'))
FIG_DIR_QC.mkdir(parents=True, exist_ok=True)
OUTDIR_QC.mkdir(parents=True, exist_ok=True)

GENE_QC = "SLC7A5"
TARGET_CELLTYPES_QC = ["Hepatocytes", "Stellete", "Myeloid"]
PLOT_LABELS_QC = {"Hepatocytes": "Hepatocytes", "Stellete": "Stellate", "Myeloid": "Myeloid"}
TOP_BOTTOM_PCT_QC = 0.20
MIN_CELLS_PER_GROUP_QC = 25

adata_qc = sc.read_h5ad(H5AD_PATH_QC)
print(adata_qc)
print("raw exists:", adata_qc.raw is not None)


In [ ]:
# Compute detection rate, top-20% group positivity, and SLC7A5 high-vs-low adjusted p-value.
def get_gene_vector_qc(adata_obj, gene):
    gene_upper = gene.upper()
    if adata_obj.raw is not None:
        raw_var_upper = pd.Index([str(g).upper() for g in adata_obj.raw.var_names])
        if gene_upper in raw_var_upper:
            raw_gene = adata_obj.raw.var_names[raw_var_upper.get_loc(gene_upper)]
            x = adata_obj.raw[:, raw_gene].X
            return np.asarray(x.toarray()).ravel() if sparse.issparse(x) else np.asarray(x).ravel()

    var_upper = pd.Index([str(g).upper() for g in adata_obj.var_names])
    if gene_upper not in var_upper:
        raise ValueError(f"{gene} was not found in adata.var_names or adata.raw.var_names")
    gene_name = adata_obj.var_names[var_upper.get_loc(gene_upper)]
    x = adata_obj[:, gene_name].X
    return np.asarray(x.toarray()).ravel() if sparse.issparse(x) else np.asarray(x).ravel()

adata_qc.obs[f"{GENE_QC}_expr_raw"] = get_gene_vector_qc(adata_qc, GENE_QC)
qc_rows = []

for ct in TARGET_CELLTYPES_QC:
    sub = adata_qc[adata_qc.obs["celltype"].astype(str) == ct].copy()
    n_total = sub.n_obs
    if n_total == 0:
        qc_rows.append({"celltype": ct, "plot_label": PLOT_LABELS_QC.get(ct, ct), "total_cells": 0})
        continue

    expr = sub.obs[f"{GENE_QC}_expr_raw"].to_numpy()
    n_positive = int((expr > 0).sum())
    detection_pct = 100 * n_positive / n_total
    n_group = int(np.floor(n_total * TOP_BOTTOM_PCT_QC))

    slc7a5_pval = np.nan
    slc7a5_adj_pvalue = np.nan
    top20_pct_positive = np.nan
    low20_pct_positive = np.nan
    high_expr_min = np.nan
    low_expr_max = np.nan

    if n_group >= MIN_CELLS_PER_GROUP_QC:
        order = np.argsort(expr, kind="mergesort")
        low_idx = order[:n_group]
        high_idx = order[-n_group:]
        low20_pct_positive = 100 * float((expr[low_idx] > 0).mean())
        top20_pct_positive = 100 * float((expr[high_idx] > 0).mean())
        low_expr_max = float(expr[low_idx].max())
        high_expr_min = float(expr[high_idx].min())

        group = np.full(n_total, "middle", dtype=object)
        group[low_idx] = "SLC7A5_low"
        group[high_idx] = "SLC7A5_high"
        sub.obs["SLC7A5_top20_bottom20_group"] = pd.Categorical(
            group,
            categories=["SLC7A5_low", "middle", "SLC7A5_high"],
            ordered=True,
        )
        sub_tb = sub[sub.obs["SLC7A5_top20_bottom20_group"] != "middle"].copy()
        sub_tb.obs["SLC7A5_top20_bottom20_group"] = sub_tb.obs["SLC7A5_top20_bottom20_group"].cat.remove_unused_categories()

        sc.tl.rank_genes_groups(
            sub_tb,
            groupby="SLC7A5_top20_bottom20_group",
            groups=["SLC7A5_high"],
            reference="SLC7A5_low",
            method="wilcoxon",
            use_raw=True,
            pts=True,
        )
        deg = sc.get.rank_genes_groups_df(sub_tb, group="SLC7A5_high").rename(columns={"names": "gene"})
        slc = deg.loc[deg["gene"].astype(str).str.upper().eq(GENE_QC)]
        if not slc.empty:
            slc7a5_pval = float(pd.to_numeric(slc.iloc[0]["pvals"], errors="coerce"))
            slc7a5_adj_pvalue = float(pd.to_numeric(slc.iloc[0]["pvals_adj"], errors="coerce"))

    qc_rows.append({
        "celltype": ct,
        "plot_label": PLOT_LABELS_QC.get(ct, ct),
        "total_cells": n_total,
        "slc7a5_positive_cells": n_positive,
        "slc7a5_detection_pct": detection_pct,
        "n_per_top_bottom_group": n_group,
        "low20_pct_positive": low20_pct_positive,
        "top20_pct_positive": top20_pct_positive,
        "low20_expr_max": low_expr_max,
        "top20_expr_min": high_expr_min,
        "slc7a5_high_vs_low_pvalue": slc7a5_pval,
        "slc7a5_high_vs_low_adjusted_pvalue": slc7a5_adj_pvalue,
    })

slc7a5_qc_df = pd.DataFrame(qc_rows)
slc7a5_qc_df["minus_log10_adjusted_pvalue"] = -np.log10(
    slc7a5_qc_df["slc7a5_high_vs_low_adjusted_pvalue"].clip(lower=1e-300)
)
qc_csv = OUTDIR_QC / "SLC7A5_stratification_feasibility_Hepatocyte_Stellate_Myeloid.csv"
slc7a5_qc_df.to_csv(qc_csv, index=False)
print(f"Saved QC table -> {qc_csv}")
display(slc7a5_qc_df)


In [ ]:
# Plot three QC panels for supplementary figure.
plot_df = slc7a5_qc_df.copy()
order = [PLOT_LABELS_QC[ct] for ct in TARGET_CELLTYPES_QC]
plot_df["plot_label"] = pd.Categorical(plot_df["plot_label"], categories=order, ordered=True)
plot_df = plot_df.sort_values("plot_label")

fig, axes = plt.subplots(1, 3, figsize=(10.8, 3.4))
palette = ["#7f7f7f", "#5b8cc0", "#b54a4a"]

sns.barplot(data=plot_df, x="plot_label", y="slc7a5_detection_pct", order=order, palette=palette, ax=axes[0])
axes[0].set_title("SLC7A5 detection")
axes[0].set_xlabel("")
axes[0].set_ylabel("SLC7A5+ cells (%)")
axes[0].set_ylim(0, max(5, plot_df["slc7a5_detection_pct"].max() * 1.25))
for i, row in plot_df.reset_index(drop=True).iterrows():
    axes[0].text(i, row["slc7a5_detection_pct"], f"n={int(row['slc7a5_positive_cells'])}", ha="center", va="bottom", fontsize=8)

sns.barplot(data=plot_df, x="plot_label", y="top20_pct_positive", order=order, palette=palette, ax=axes[1])
axes[1].set_title("Top 20% group positivity")
axes[1].set_xlabel("")
axes[1].set_ylabel("SLC7A5+ cells in top 20% (%)")
axes[1].set_ylim(0, max(5, plot_df["top20_pct_positive"].max() * 1.25))
for i, row in plot_df.reset_index(drop=True).iterrows():
    axes[1].text(i, row["top20_pct_positive"], f"n/group={int(row['n_per_top_bottom_group'])}", ha="center", va="bottom", fontsize=8)

sns.barplot(data=plot_df, x="plot_label", y="minus_log10_adjusted_pvalue", order=order, palette=palette, ax=axes[2])
axes[2].axhline(-np.log10(0.05), color="black", linestyle="--", linewidth=0.9)
axes[2].set_title("SLC7A5 high vs low")
axes[2].set_xlabel("")
axes[2].set_ylabel("-log10(adjusted p-value)")
axes[2].set_ylim(0, max(2, plot_df["minus_log10_adjusted_pvalue"].max() * 1.18))
for i, row in plot_df.reset_index(drop=True).iterrows():
    adj = row["slc7a5_high_vs_low_adjusted_pvalue"]
    label = "adj p=NA" if pd.isna(adj) else f"adj p={adj:.2g}"
    axes[2].text(i, row["minus_log10_adjusted_pvalue"] if pd.notna(row["minus_log10_adjusted_pvalue"]) else 0, label, ha="center", va="bottom", fontsize=8)

for ax in axes:
    ax.tick_params(axis="x", rotation=30)
    sns.despine(ax=ax)

plt.suptitle("SLC7A5 stratification feasibility across liver cell types", y=1.04)
plt.tight_layout()

qc_pdf = FIG_DIR_QC / "Suppl_SLC7A5_stratification_feasibility_Hepatocyte_Stellate_Myeloid.pdf"
qc_png = FIG_DIR_QC / "Suppl_SLC7A5_stratification_feasibility_Hepatocyte_Stellate_Myeloid.png"
plt.savefig(qc_pdf, bbox_inches="tight", dpi=300)
plt.savefig(qc_png, bbox_inches="tight", dpi=200)
plt.show()

print(f"Saved -> {qc_pdf}")
print(f"Saved -> {qc_png}")


# Fig1E revised. Myeloid SLC7A5-high vs SLC7A5-low GSEA

Myeloid cells are split into SLC7A5 top 20% and bottom 20%. DEG is calculated with raw expression (`use_raw=True`), then preranked GSEA is run for Hallmark and KEGG. The focused output summarizes Glycolysis, OXPHOS, TCA cycle, FAO, NF-kB, TLR-LPS, and mTOR-related pathways.

In [ ]:
# Fig1E revised setup: Myeloid-only SLC7A5 top/bottom 20% DEG
import re
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
import gseapy as gp
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import sparse

H5AD_PATH = artifact_path('02_SLC7A5_mono_Journal/03_Script/Journal_script/scRNA_monoMAC_detail.h5ad')
FIG_DIR = Path(output_dir('02_SLC7A5_mono_Journal/05_Figure'))
OUTDIR = Path(output_dir('02_SLC7A5_mono_Journal/02_Analysis/Fig1E_Myeloid_SLC7A5_top20_bottom20_GSEA'))
FIG_DIR.mkdir(parents=True, exist_ok=True)
OUTDIR.mkdir(parents=True, exist_ok=True)

GENE = "SLC7A5"
CELLTYPE = "Myeloid"
TOP_BOTTOM_PCT = 0.20
MIN_CELLS_PER_GROUP = 25
LFC_CAP = 10.0
EXCLUDE_STRATIFICATION_GENE_FROM_GSEA = True

adata_my = sc.read_h5ad(H5AD_PATH)
print(adata_my)
print("raw exists:", adata_my.raw is not None)
print(adata_my.obs["celltype"].value_counts())


In [ ]:
# Build Myeloid SLC7A5 high/low groups from raw expression.
def get_gene_vector(adata_obj, gene):
    gene_upper = gene.upper()
    if adata_obj.raw is not None:
        raw_var_upper = pd.Index([str(g).upper() for g in adata_obj.raw.var_names])
        if gene_upper in raw_var_upper:
            raw_gene = adata_obj.raw.var_names[raw_var_upper.get_loc(gene_upper)]
            x = adata_obj.raw[:, raw_gene].X
            return np.asarray(x.toarray()).ravel() if sparse.issparse(x) else np.asarray(x).ravel()

    var_upper = pd.Index([str(g).upper() for g in adata_obj.var_names])
    if gene_upper not in var_upper:
        raise ValueError(f"{gene} was not found in adata.var_names or adata.raw.var_names")
    gene_name = adata_obj.var_names[var_upper.get_loc(gene_upper)]
    x = adata_obj[:, gene_name].X
    return np.asarray(x.toarray()).ravel() if sparse.issparse(x) else np.asarray(x).ravel()

adata_my.obs[f"{GENE}_expr_raw"] = get_gene_vector(adata_my, GENE)
sub = adata_my[adata_my.obs["celltype"].astype(str) == CELLTYPE].copy()

n_total = sub.n_obs
n_group = int(np.floor(n_total * TOP_BOTTOM_PCT))
if n_group < MIN_CELLS_PER_GROUP:
    raise ValueError(f"Not enough {CELLTYPE} cells for top/bottom split: n_total={n_total}, n_group={n_group}")

expr = sub.obs[f"{GENE}_expr_raw"].to_numpy()
order = np.argsort(expr, kind="mergesort")
group = np.full(n_total, "middle", dtype=object)
group[order[:n_group]] = "SLC7A5_low"
group[order[-n_group:]] = "SLC7A5_high"
sub.obs["SLC7A5_top20_bottom20_group"] = pd.Categorical(
    group,
    categories=["SLC7A5_low", "middle", "SLC7A5_high"],
    ordered=True,
)

sub_tb = sub[sub.obs["SLC7A5_top20_bottom20_group"] != "middle"].copy()
sub_tb.obs["SLC7A5_top20_bottom20_group"] = sub_tb.obs["SLC7A5_top20_bottom20_group"].cat.remove_unused_categories()

split_summary = pd.DataFrame([
    {
        "celltype": CELLTYPE,
        "total_cells": n_total,
        "n_per_group": n_group,
        "low_expr_min": float(expr[order[:n_group]].min()),
        "low_expr_max": float(expr[order[:n_group]].max()),
        "high_expr_min": float(expr[order[-n_group:]].min()),
        "high_expr_max": float(expr[order[-n_group:]].max()),
        "low_pct_positive": float((expr[order[:n_group]] > 0).mean()),
        "high_pct_positive": float((expr[order[-n_group:]] > 0).mean()),
    }
])
split_summary.to_csv(OUTDIR / "Myeloid_SLC7A5_top20_bottom20_split_summary.csv", index=False)
display(split_summary)


In [ ]:
# DEG: Myeloid SLC7A5-high vs SLC7A5-low. Use raw to avoid scaled-X logFC artifacts.
sc.tl.rank_genes_groups(
    sub_tb,
    groupby="SLC7A5_top20_bottom20_group",
    groups=["SLC7A5_high"],
    reference="SLC7A5_low",
    method="wilcoxon",
    use_raw=True,
    pts=True,
)

deg_myeloid = sc.get.rank_genes_groups_df(sub_tb, group="SLC7A5_high").rename(columns={"names": "gene"})
for col in ["scores", "logfoldchanges", "pvals", "pvals_adj"]:
    deg_myeloid[col] = pd.to_numeric(deg_myeloid[col], errors="coerce")
deg_myeloid["logfoldchanges"] = deg_myeloid["logfoldchanges"].clip(-LFC_CAP, LFC_CAP)
deg_myeloid.insert(0, "celltype", CELLTYPE)

deg_path = OUTDIR / "DEG_Myeloid_SLC7A5_top20_high_vs_bottom20_low.csv"
deg_myeloid.to_csv(deg_path, index=False)
print(f"Saved DEG -> {deg_path}")
display(deg_myeloid.head(20))
display(deg_myeloid.loc[deg_myeloid["gene"].astype(str).str.upper().eq(GENE)])


In [ ]:
# Preranked GSEA: Hallmark and KEGG. Focus on Glycolysis, OXPHOS, TCA cycle, FAO, NF-kB, TLR-LPS, and mTOR pathways.
GENE_SET_COLLECTIONS = {
    "Hallmark": "MSigDB_Hallmark_2020",
    "KEGG": "KEGG_2021_Human",
}

FOCUS_PATTERNS = {
    "Glycolysis": r"GLYCOLYSIS|GLUCONEOGENESIS",
    "OXPHOS": r"OXIDATIVE_PHOSPHORYLATION|RESPIRATORY_CHAIN|ELECTRON_TRANSPORT_CHAIN",
    "TCA cycle": r"TCA|CITRATE_CYCLE|TRICARBOXYLIC|CARBON_METABOLISM|PYRUVATE_METABOLISM",
    "FAO": r"FATTY_ACID|BETA_OXIDATION|ACYL_COA|PPAR_SIGNALING",
    "NF-kB": r"NFKB|NF_KB|TNFA_SIGNALING_VIA_NFKB",
    "TLR-LPS": r"TOLL_LIKE_RECEPTOR|LIPOPOLYSACCHARIDE|LPS|NOD_LIKE_RECEPTOR",
    "mTOR": r"MTOR|PI3K_AKT_MTOR|MTORC1",
}

def build_prerank(deg):
    rnk = deg[["gene", "logfoldchanges", "pvals", "scores"]].copy()
    rnk["gene"] = rnk["gene"].astype(str).str.upper()
    if EXCLUDE_STRATIFICATION_GENE_FROM_GSEA:
        rnk = rnk.loc[rnk["gene"] != GENE]
    rnk["pvals"] = pd.to_numeric(rnk["pvals"], errors="coerce").clip(lower=1e-300)
    rnk["logfoldchanges"] = pd.to_numeric(rnk["logfoldchanges"], errors="coerce")
    rnk["scores"] = pd.to_numeric(rnk["scores"], errors="coerce")
    rnk["rank_metric"] = np.sign(rnk["logfoldchanges"]) * -np.log10(rnk["pvals"])
    rnk.loc[rnk["rank_metric"].isna(), "rank_metric"] = rnk.loc[rnk["rank_metric"].isna(), "scores"]
    rnk = rnk.replace([np.inf, -np.inf], np.nan).dropna(subset=["gene", "rank_metric"])
    rnk = rnk.sort_values("rank_metric", ascending=False).drop_duplicates("gene", keep="first")
    return rnk[["gene", "rank_metric"]]

prerank_df = build_prerank(deg_myeloid)
rnk_path = OUTDIR / "Prerank_Myeloid_SLC7A5_top20_high_vs_low.rnk"
prerank_df.to_csv(rnk_path, sep="	", index=False, header=False)
print(f"Saved prerank -> {rnk_path} | n_genes={len(prerank_df)}")

gsea_tables = []
for collection_label, gene_sets in GENE_SET_COLLECTIONS.items():
    try:
        pre_res = gp.prerank(
            rnk=prerank_df,
            gene_sets=gene_sets,
            min_size=5,
            max_size=500,
            permutation_num=1000,
            seed=7,
            outdir=None,
            verbose=False,
        )
    except Exception as e:
        print(f"[{collection_label}] failed with {gene_sets}: {e}")
        continue

    res = pre_res.res2d.copy()
    res.insert(0, "collection", collection_label)
    res.insert(1, "celltype", CELLTYPE)
    res_path = OUTDIR / f"GSEA_Myeloid_{collection_label}_SLC7A5_top20_high_vs_low.csv"
    res.to_csv(res_path, index=False)
    gsea_tables.append(res)
    print(f"[{collection_label}] saved -> {res_path} | terms={len(res)}")

if not gsea_tables:
    raise RuntimeError("No GSEA results were generated. Check gseapy gene set library access.")

gsea_all = pd.concat(gsea_tables, ignore_index=True)
gsea_all.to_csv(OUTDIR / "GSEA_Myeloid_all_collections_SLC7A5_top20_high_vs_low.csv", index=False)

focused = []
term_norm = gsea_all["Term"].astype(str).str.upper().str.replace(" ", "_", regex=False).str.replace("-", "_", regex=False)
for pathway_group, pattern in FOCUS_PATTERNS.items():
    hit = gsea_all.loc[term_norm.str.contains(pattern, regex=True, na=False)].copy()
    if hit.empty:
        print(f"No matched terms for {pathway_group}")
        continue
    hit.insert(2, "pathway_group", pathway_group)
    focused.append(hit)

focused_gsea = pd.concat(focused, ignore_index=True) if focused else pd.DataFrame()
if not focused_gsea.empty:
    focused_gsea["adjusted_pvalue"] = pd.to_numeric(focused_gsea["FDR q-val"], errors="coerce")
focused_gsea.to_csv(OUTDIR / "GSEA_Myeloid_focused_Glycolysis_OXPHOS_TCA_FAO_NFKB_TLRLPS_mTOR.csv", index=False)
print(f"Focused matched terms: {len(focused_gsea)}")
display(focused_gsea[["collection", "pathway_group", "Term", "NES", "NOM p-val", "adjusted_pvalue"]].head(50))


In [ ]:
# Save focused GSEA PDF figures.
if focused_gsea.empty:
    raise ValueError("No focused GSEA terms matched. Inspect GSEA_Myeloid_all_collections_SLC7A5_top20_high_vs_low.csv manually.")

plot_df = focused_gsea.copy()
for col in ["NES", "NOM p-val", "adjusted_pvalue"]:
    plot_df[col] = pd.to_numeric(plot_df[col], errors="coerce")
plot_df["abs_NES"] = plot_df["NES"].abs()

# Keep one representative term per pathway group and collection, prioritizing adjusted p-value then absolute NES.
best_df = (
    plot_df.sort_values(["pathway_group", "collection", "adjusted_pvalue", "abs_NES"], ascending=[True, True, True, False])
    .groupby(["pathway_group", "collection"], observed=True, as_index=False)
    .head(1)
    .copy()
)
best_df["pathway_group"] = pd.Categorical(
    best_df["pathway_group"],
    categories=["Glycolysis", "OXPHOS", "TCA cycle", "FAO", "NF-kB", "TLR-LPS", "mTOR"],
    ordered=True,
)
best_df = best_df.sort_values(["pathway_group", "collection"])
best_df["label"] = best_df["pathway_group"].astype(str) + " | " + best_df["collection"].astype(str)
best_df["term_short"] = best_df["Term"].astype(str).str.replace("HALLMARK_", "", regex=False).str.replace("KEGG_", "", regex=False)
best_df.to_csv(OUTDIR / "GSEA_Myeloid_focused_best_terms_for_pdf.csv", index=False)

fig_height = max(4.5, 0.42 * len(best_df) + 1.5)
fig, ax = plt.subplots(figsize=(8.5, fig_height))
colors = ["#b2182b" if v > 0 else "#2166ac" for v in best_df["NES"]]
ax.barh(best_df["label"], best_df["NES"], color=colors, alpha=0.88)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("NES (SLC7A5-high vs SLC7A5-low Myeloid)")
ax.set_ylabel("")
ax.set_title("Fig1E. Myeloid SLC7A5-associated pathway enrichment")
ax.invert_yaxis()

x_min, x_max = ax.get_xlim()
x_span = x_max - x_min
for y, (_, row) in enumerate(best_df.iterrows()):
    star = "**" if row["adjusted_pvalue"] < 0.01 else "*" if row["adjusted_pvalue"] < 0.05 else ""
    txt = f"{star} adj p={row['adjusted_pvalue']:.3g}; {row['term_short']}"
    x = row["NES"] + (0.02 * x_span if row["NES"] >= 0 else -0.02 * x_span)
    ha = "left" if row["NES"] >= 0 else "right"
    ax.text(x, y, txt, va="center", ha=ha, fontsize=7)

plt.tight_layout()
bar_pdf = FIG_DIR / "Fig1E_Myeloid_SLC7A5_top20_bottom20_Hallmark_KEGG_focused_GSEA_barplot.pdf"
bar_png = FIG_DIR / "Fig1E_Myeloid_SLC7A5_top20_bottom20_Hallmark_KEGG_focused_GSEA_barplot.png"
plt.savefig(bar_pdf, bbox_inches="tight", dpi=300)
plt.savefig(bar_png, bbox_inches="tight", dpi=200)
plt.show()

# Dot plot with all focused matched terms, capped to the most significant terms for readability.
dot_df = plot_df.sort_values(["pathway_group", "adjusted_pvalue", "abs_NES"], ascending=[True, True, False]).groupby("pathway_group", observed=True).head(8).copy()
dot_df["minus_log10_adjusted_pvalue"] = -np.log10(dot_df["adjusted_pvalue"].clip(lower=1e-300))
dot_df["term_label"] = dot_df["collection"] + " | " + dot_df["Term"].astype(str).str.replace("HALLMARK_", "", regex=False).str.replace("KEGG_", "", regex=False)

fig_height = max(5, 0.28 * len(dot_df) + 1.6)
fig, ax = plt.subplots(figsize=(9.5, fig_height))
scat = ax.scatter(
    dot_df["NES"],
    dot_df["term_label"],
    c=dot_df["NES"],
    s=20 + 28 * dot_df["minus_log10_adjusted_pvalue"].clip(upper=10),
    cmap="RdBu_r",
    vmin=-max(1, dot_df["NES"].abs().max()),
    vmax=max(1, dot_df["NES"].abs().max()),
    edgecolor="black",
    linewidth=0.25,
)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("NES (SLC7A5-high vs SLC7A5-low Myeloid)")
ax.set_ylabel("")
ax.set_title("Focused GSEA terms: Glycolysis, OXPHOS, TCA cycle, FAO, NF-kB, TLR-LPS, mTOR")
cbar = plt.colorbar(scat, ax=ax, pad=0.02)
cbar.set_label("NES")
plt.tight_layout()
dot_pdf = FIG_DIR / "Fig1E_Myeloid_SLC7A5_top20_bottom20_Hallmark_KEGG_focused_GSEA_dotplot.pdf"
dot_png = FIG_DIR / "Fig1E_Myeloid_SLC7A5_top20_bottom20_Hallmark_KEGG_focused_GSEA_dotplot.png"
plt.savefig(dot_pdf, bbox_inches="tight", dpi=300)
plt.savefig(dot_png, bbox_inches="tight", dpi=200)
plt.show()

print(f"Saved -> {bar_pdf}")
print(f"Saved -> {dot_pdf}")
print(f"Tables -> {OUTDIR}")
